# Error Analysis Template

Error analysis means studying where the model fails. In tyre inspection, false negatives matter a lot because they mean a real defect may be missed.

This notebook is a reusable template. It uses currently available tracked artifacts where they exist and shows clear pending sections where they do not.

## Why This Matters
### Simple explanation
A single score like accuracy does not explain what kind of mistakes the model makes.

### Technical detail
Industrial inspection is often asymmetric. Missing a defect can be much worse than flagging a good tyre for review, so confusion matrices and false-negative review are central.

In [1]:
from pathlib import Path
import json
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the TyreVisionX repo root from the current working directory.')

repo_root = find_repo_root()
print('Repo root:', repo_root)
candidate_run = repo_root / 'artifacts/day3_baseline/frozen_resnet50_unfreeze_v2_3ep'
print('Using artifact directory:', candidate_run)
print('Exists:', candidate_run.exists())

Repo root: /Users/ritik/Documents/Project TDA/TyreVisionX
Using artifact directory: /Users/ritik/Documents/Project TDA/TyreVisionX/artifacts/day3_baseline/frozen_resnet50_unfreeze_v2_3ep
Exists: True


In [2]:
eval_report_path = candidate_run / 'eval_report.json'
if eval_report_path.exists():
    report = json.loads(eval_report_path.read_text(encoding='utf-8'))
    print('Metrics:')
    for key, value in report.get('metrics', {}).items():
        print(f'  {key}: {value}')
    print('Confusion:', report.get('metrics', {}).get('confusion', 'see report structure'))
else:
    print('Pending: eval report not found for selected artifact directory.')

Metrics:
  accuracy: 0.9803921580314636
  precision: 0.9804379940032959
  recall: 0.9806153774261475
  f1_macro: 0.9803909063339233
  precision_defect: 0.9921259880065918
  recall_defect: 0.9692307710647583
  f1_defect: 0.9805447459220886
  auroc: 0.9985231161117554
  auprc: 0.9985852837562561
  confusion: [[124, 1], [4, 126]]
Confusion: [[124, 1], [4, 126]]


In [3]:
fn_path = candidate_run / 'false_negatives.csv'
mis_path = candidate_run / 'misclassified.csv'

if fn_path.exists():
    fn_df = pd.read_csv(fn_path)
    print('False negatives:', len(fn_df))
    display(fn_df.head(10))
else:
    print('Pending: false_negatives.csv not available.')

if mis_path.exists():
    mis_df = pd.read_csv(mis_path)
    print('Misclassified rows:', len(mis_df))
    display(mis_df.head(10))
else:
    print('Pending: misclassified.csv not available.')

False negatives: 4


,image_path,dataset_id,split,target,pred,target_label,pred_label,prob_good,prob_defect,is_misclassified,is_false_negative
0,/Users/ritik/Documents/Project TDA/TyreVisionX...,D1,test,1,0,defect,good,0.700321,0.299679,1,1
1,/Users/ritik/Documents/Project TDA/TyreVisionX...,D1,test,1,0,defect,good,0.747228,0.252772,1,1
2,/Users/ritik/Documents/Project TDA/TyreVisionX...,D1,test,1,0,defect,good,0.606068,0.393932,1,1
3,/Users/ritik/Documents/Project TDA/TyreVisionX...,D1,test,1,0,defect,good,0.801382,0.198618,1,1


Misclassified rows: 5


,image_path,dataset_id,split,target,pred,target_label,pred_label,prob_good,prob_defect,is_misclassified,is_false_negative
0,/Users/ritik/Documents/Project TDA/TyreVisionX...,D1,test,1,0,defect,good,0.700321,0.299679,1,1
1,/Users/ritik/Documents/Project TDA/TyreVisionX...,D1,test,1,0,defect,good,0.747228,0.252772,1,1
2,/Users/ritik/Documents/Project TDA/TyreVisionX...,D1,test,0,1,good,defect,0.144470,0.855530,1,0
3,/Users/ritik/Documents/Project TDA/TyreVisionX...,D1,test,1,0,defect,good,0.606068,0.393932,1,1
4,/Users/ritik/Documents/Project TDA/TyreVisionX...,D1,test,1,0,defect,good,0.801382,0.198618,1,1


## Discussion Prompts
- Are the false negatives visually subtle?
- Are there signs of label noise or ambiguous samples?
- Does the model fail under lighting, angle, blur, or background changes?
- Is the operating threshold aligned with defect-sensitive deployment goals?

## Pending Canonical Error Analysis
Pending items after the cleanup refactor:
- canonical-pipeline false-negative review
- canonical-pipeline false-positive review
- D2 and D3 error comparison
- threshold-calibration study under the cleaned path

## Future Work Sections
### Anomaly detection
Could help when defects are rare, diverse, or not fully labeled.

### Localization
Would let reviewers see where the model believes the defect is.

### Segmentation
Would provide pixel-level error analysis instead of only image-level analysis.

### Multi-view 3D
Would allow disagreement analysis across views of the same tyre.

### Knowledge reasoning
Could connect visual errors to defect taxonomy, probable causes, and inspection workflow decisions.